In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import copy
import csv
import hashlib
import json
import platform
import random
import time
from pathlib import Path
import re

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================
RUN_PROFILE = "paper"  # "smoke", "paper", "repo_cli"

METHODS_TO_RUN = [
    "boundary_shrink",
    "boundary_expanding"
]

FORGET_CLASS = 4
SEED = 2022
BATCH_SIZE = 64
NUM_WORKERS = 2

REUSE_CHECKPOINTS = False
DOWNLOAD_CIFAR10 = True  # Official labeled test set; cached under /kaggle/working.
ATTACK_NORMALIZATION = "repo_exact"  # Preserve the author's attack normalization.

FGSM_BOUND = 0.1
UNLEARN_LR = 1e-5
MOMENTUM = 0.9
NUM_CLASSES = 10
# Optional Enhanced Boundary Shrink, matching the author's defaults.
EXTRA_EXP = None  # None, "curv", or "weight_assign"
CURVATURE_LAMBDA = 0.7
REWEIGHT_BIAS = -0.5
REWEIGHT_SLOPE = 5.0
# ============================================================
# 2. TRAINING CONFIGURATION
# ============================================================
if RUN_PROFILE == "smoke":
    ORIGINAL_EPOCHS = 1
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 1
    MAX_BATCHES_PER_EPOCH = 3
    EVALUATE_TRAIN_SPLITS = False

elif RUN_PROFILE == "paper":
    ORIGINAL_EPOCHS = 30
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

elif RUN_PROFILE == "repo_cli":
    ORIGINAL_EPOCHS = 50
    ORIGINAL_LR = 0.01
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

else:
    raise ValueError(
        "RUN_PROFILE must be smoke, paper, or repo_cli"
    )

if ATTACK_NORMALIZATION not in {"matched", "repo_exact"}:
    raise ValueError("Unsupported ATTACK_NORMALIZATION")

if EXTRA_EXP not in {None, "curv", "weight_assign"}:
    raise ValueError("Unsupported EXTRA_EXP")

if not 0 <= FORGET_CLASS < NUM_CLASSES:
    raise ValueError("FORGET_CLASS must be between 0 and 9")

if any(
    method not in {"boundary_shrink", "boundary_expanding"}
    for method in METHODS_TO_RUN
):
    raise ValueError("Unsupported unlearning method")

# ============================================================
# 3. KAGGLE PATHS
# ============================================================
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = Path(
    "/kaggle/input/datasets/petitbonney/cifar10-image-recognition"
)

OUTPUT_ROOT = Path("/kaggle/working")
CIFAR10_ROOT = OUTPUT_ROOT / "cifar10"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
RESULT_ROOT = OUTPUT_ROOT / "results"

# Kaggle Input is read-only; create directories only in working.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 4. NORMALIZATION
# ============================================================
CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)

REPO_ATTACK_MEAN = (0.4914, 0.4822, 0.2265)
REPO_ATTACK_STD = (0.2023, 0.1994, 0.2010)

# ============================================================
# 5. REPRODUCIBILITY
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# 6. ENVIRONMENT CHECK
# ============================================================
print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is unavailable. Enable GPU in Kaggle settings.")

print("\nInput directories:")
for path in INPUT_ROOT.iterdir():
    print(" -", path)

print("\nData root exists:", DATA_ROOT.exists())
print("Output root:", OUTPUT_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Result root:", RESULT_ROOT)
print("\nConfiguration initialized successfully.")


In [ ]:
import re
import csv
from pathlib import Path
import numpy as np

# ============================================================
# 1. CIFAR-10 CLASSES
# ============================================================
CIFAR_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

CLASS_TO_INDEX = {
    name: index for index, name in enumerate(CIFAR_CLASSES)
}

# ============================================================
# 2. LOAD IMAGE ARRAYS
# ============================================================

def load_concatenated_npy_images(path):
    path = Path(path)
    with path.open("rb") as stream:
        version = np.lib.format.read_magic(stream)
        if version == (1, 0):
            shape, fortran_order, dtype = np.lib.format.read_array_header_1_0(stream)
        elif version == (2, 0):
            shape, fortran_order, dtype = np.lib.format.read_array_header_2_0(stream)
        else:
            raise ValueError(f"Unsupported NPY version: {version}")
        header_size = stream.tell()
        stream.seek(0)
        first_header = stream.read(header_size)

    if dtype != np.dtype(np.uint8) or fortran_order:
        raise ValueError(f"Expected C-order uint8 images, got dtype={dtype}, fortran_order={fortran_order}")
    if shape not in {(32, 32, 3), (3, 32, 32), (3072,)}:
        raise ValueError(f"Unsupported image record shape: {shape}")

    image_size = int(np.prod(shape)) * dtype.itemsize
    record_size = header_size + image_size
    file_size = path.stat().st_size
    if file_size % record_size != 0:
        raise ValueError(f"Unexpected file size: {file_size}")
    num_images = file_size // record_size
    raw = np.memmap(path, dtype=np.uint8, mode="r")
    for i in sorted({0, min(1, num_images - 1), num_images - 1}):
        offset = i * record_size
        if bytes(raw[offset:offset + header_size]) != first_header:
            raise ValueError(f"Inconsistent NPY header at record {i}")

    # Preserve the image layout declared in each record (HWC for this dataset).
    image_strides = np.empty(shape, dtype=dtype).strides
    images = np.ndarray(
        shape=(num_images, *shape), dtype=dtype, buffer=raw,
        offset=header_size, strides=(record_size, *image_strides),
    )
    return images, raw

train_images, train_raw = load_concatenated_npy_images(DATA_ROOT / "train.npy")
test_images, test_raw = load_concatenated_npy_images(DATA_ROOT / "test.npy")
print("Train images:", train_images.shape, train_images.dtype)
print("Test images:", test_images.shape, test_images.dtype)


# ============================================================
# 3. READ CSV FILE
# ============================================================
def read_csv_rows(path):
    path = Path(path)

    with path.open(
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as stream:
        reader = csv.DictReader(stream)

        if reader.fieldnames is None:
            raise ValueError(
                f"CSV file has no header: {path}"
            )

        columns = {
            name.strip(): name
            for name in reader.fieldnames
            if name is not None
        }

        rows = list(reader)

    if not rows:
        raise ValueError(f"CSV file is empty: {path}")

    return rows, columns


# ============================================================
# 4. VALIDATE REQUIRED COLUMN
# ============================================================
def required_column(columns, name, path):
    if name not in columns:
        raise ValueError(
            f"Required column '{name}' not found in {path}. "
            f"Available columns: {list(columns.keys())}"
        )

    return columns[name]


# ============================================================
# 5. ENCODE CIFAR-10 LABELS
# ============================================================
def encode_cifar_labels(raw_labels):
    labels_as_text = [
        str(value).strip().lower()
        for value in raw_labels
    ]

    numeric_values = []
    all_numeric = True

    for value in labels_as_text:
        try:
            numeric_values.append(int(value))
        except (ValueError, TypeError):
            all_numeric = False
            break

    if all_numeric:
        unique = set(numeric_values)

        # Labels from 0 to 9
        if unique.issubset(set(range(NUM_CLASSES))):
            return np.asarray(numeric_values, dtype=np.int64)

        # Labels from 1 to 10
        if unique.issubset(set(range(1, NUM_CLASSES + 1))):
            return np.asarray(
                numeric_values, dtype=np.int64
            ) - 1

        raise ValueError(
            f"Invalid numeric CIFAR-10 labels: {sorted(unique)}"
        )

    unknown = sorted(
        set(labels_as_text) - set(CLASS_TO_INDEX.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown CIFAR-10 class labels: {unknown}"
        )

    return np.asarray(
        [CLASS_TO_INDEX[value] for value in labels_as_text],
        dtype=np.int64
    )


# ============================================================
# 6. PARSE NUMERIC ID
# ============================================================
def parse_numeric_id(value):
    text = str(value).strip()
    match = re.search(r"\d+", text)

    return int(match.group()) if match else None





In [ ]:
train_label_rows, train_label_columns = read_csv_rows(DATA_ROOT / "trainLabels.csv")
train_label_col = required_column(train_label_columns, "label", DATA_ROOT / "trainLabels.csv")
train_id_col = required_column(train_label_columns, "id", DATA_ROOT / "trainLabels.csv")
train_labels = encode_cifar_labels([row[train_label_col] for row in train_label_rows])
train_ids = [parse_numeric_id(row[train_id_col]) for row in train_label_rows]
if all(value is None for value in train_ids):
    print("Warning: No numeric IDs found in training labels.")

sample_rows, sample_columns = read_csv_rows(DATA_ROOT / "sampleSubmission.csv")
sample_id_col = required_column(sample_columns, "id", DATA_ROOT / "sampleSubmission.csv")
sample_label_col = required_column(sample_columns, "label", DATA_ROOT / "sampleSubmission.csv")
sample_ids = [parse_numeric_id(row[sample_id_col]) for row in sample_rows]
sample_labels = [row[sample_label_col] for row in sample_rows]

print("Train label columns:", list(train_label_columns.keys()))
print("Number of training labels:", len(train_labels))
print("Number of training IDs:", len(train_ids))
print("Training sample rows:", train_label_rows[:5])
print("Sample submission columns:", list(sample_columns.keys()))
print("Number of sample submission rows:", len(sample_rows))
print("Sample submission rows:", sample_rows[:5])
print("Encoded training labels:", train_labels[:5])
print("Parsed training IDs:", train_ids[:5])


In [ ]:
def validate_image_array(array, name):
    if array.ndim == 2 and array.shape[1] == 3072:
        return
    if array.ndim == 4 and array.shape[1:] in {(32, 32, 3), (3, 32, 32)}:
        return
    raise ValueError(f"{name} has unsupported shape {array.shape}. Expected (N, 3072), (N, 32, 32, 3), or (N, 3, 32, 32).")

validate_image_array(train_images, "train.npy")
validate_image_array(test_images, "test.npy")

if len(train_images) != len(train_labels):
    raise ValueError(f"Training image/label count mismatch: {len(train_images)} images vs {len(train_labels)} labels.")
if len(test_images) != len(sample_ids):
    raise ValueError(f"Test image/sample submission count mismatch: {len(test_images)} images vs {len(sample_ids)} sample IDs.")

print("Image arrays validated successfully.")
print("Train images:", train_images.shape)
print("Train labels:", train_labels.shape)
print("Test images:", test_images.shape)
print("Sample submission IDs:", len(sample_ids))


In [ ]:
class CIFAR10NpyDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels=None):
        self.images = images
        self.labels = None if labels is None else np.asarray(labels, dtype=np.int64)
        if self.labels is not None and len(self.images) != len(self.labels):
            raise ValueError("Images and labels have different lengths.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = np.asarray(self.images[index])
        if image.ndim == 1:
            if image.size != 3072:
                raise ValueError(f"Invalid flattened image shape: {image.shape}")
            image = image.reshape(3, 32, 32).transpose(1, 2, 0)
        elif image.shape == (32, 32, 3):
            pass
        elif image.shape == (3, 32, 32):
            image = image.transpose(1, 2, 0)
        else:
            raise ValueError(f"Unsupported image shape: {image.shape}")

        source_is_integer = np.issubdtype(image.dtype, np.integer)
        image = image.astype(np.float32)
        min_value = float(image.min())
        max_value = float(image.max())
        if source_is_integer or (min_value >= 0.0 and max_value > 1.0):
            image /= 255.0
        elif min_value < 0.0 or max_value > 1.0:
            raise ValueError(f"Unexpected image range: [{min_value}, {max_value}]")

        image = np.clip(image, 0.0, 1.0)
        image = np.ascontiguousarray(((image - 0.5) / 0.5).transpose(2, 0, 1), dtype=np.float32)
        image_tensor = torch.from_numpy(image)
        if self.labels is None:
            return image_tensor
        return image_tensor, int(self.labels[index])



In [ ]:
train_dataset = CIFAR10NpyDataset(train_images, train_labels)
test_inference_dataset = CIFAR10NpyDataset(test_images)

# The Kaggle competition test arrays have no ground-truth labels.
# Match the author by evaluating on the separate official CIFAR-10 test set.
evaluation_dataset = datasets.CIFAR10(
    root=str(CIFAR10_ROOT),
    train=False,
    download=DOWNLOAD_CIFAR10,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]),
)

train_fit_set = train_dataset
train_forget_indices = np.flatnonzero(train_labels == FORGET_CLASS).tolist()
train_retain_indices = np.flatnonzero(train_labels != FORGET_CLASS).tolist()
test_labels = np.asarray(evaluation_dataset.targets, dtype=np.int64)
test_forget_indices = np.flatnonzero(test_labels == FORGET_CLASS).tolist()
test_retain_indices = np.flatnonzero(test_labels != FORGET_CLASS).tolist()

train_forget_set = Subset(train_dataset, train_forget_indices)
train_retain_set = Subset(train_dataset, train_retain_indices)
test_forget_set = Subset(evaluation_dataset, test_forget_indices)
test_retain_set = Subset(evaluation_dataset, test_retain_indices)
test_inference_set = test_inference_dataset

print("Train fit:", len(train_fit_set))
print("Train forget:", len(train_forget_set))
print("Train retain:", len(train_retain_set))
print("Official test:", len(evaluation_dataset))
print("Official test forget:", len(test_forget_set))
print("Official test retain:", len(test_retain_set))
print("Kaggle test inference:", len(test_inference_set))


In [ ]:
PIN_MEMORY = DEVICE.type == "cuda"

def make_loader(dataset, shuffle=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, generator=generator, persistent_workers=False)

train_all_loader = make_loader(train_fit_set, shuffle=True)
train_retain_loader = make_loader(train_retain_set, shuffle=True)
train_retain_eval_loader = make_loader(train_retain_set, shuffle=False)
train_forget_eval_loader = make_loader(train_forget_set, shuffle=False)

test_loader = make_loader(evaluation_dataset, shuffle=False)
test_forget_loader = make_loader(test_forget_set, shuffle=False)
test_retain_loader = make_loader(test_retain_set, shuffle=False)
test_inference_loader = make_loader(test_inference_set, shuffle=False)

class_count = np.bincount(train_labels, minlength=NUM_CLASSES)

print("Class counts:", dict(zip(CIFAR_CLASSES, class_count.tolist())))
print("Train all batches:", len(train_all_loader))
print("Train retain batches:", len(train_retain_loader))
print("Train retain evaluation batches:", len(train_retain_eval_loader))
print("Train forget evaluation batches:", len(train_forget_eval_loader))
print("Official test batches:", len(test_loader))
print("Official test forget batches:", len(test_forget_loader))
print("Official test retain batches:", len(test_retain_loader))
print("Kaggle test inference batches:", len(test_inference_loader))

In [ ]:

# ============================================================
# PHASE 3: MODEL ARCHITECTURE
# ============================================================
class Identity(nn.Module):
    def forward(self, x):
        return x


class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)


class Conv(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=None, activation_fn=nn.ReLU, batch_norm=True):
        if padding is None:
            padding = (kernel_size - 1) // 2
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=not batch_norm)]
        if batch_norm:
            layers.append(nn.BatchNorm2d(out_channels, affine=True))
        layers.append(activation_fn())
        super().__init__(*layers)


class AllCNN(nn.Module):
    def __init__(self, n_channels=3, num_classes=10, filters_percentage=0.5, batch_norm=True):
        super().__init__()
        n_filters_1 = int(96 * filters_percentage)
        n_filters_2 = int(192 * filters_percentage)
        self.features = nn.Sequential(
            Conv(n_channels, n_filters_1, kernel_size=3, batch_norm=batch_norm),
            Conv(n_filters_1, n_filters_1, kernel_size=3, batch_norm=batch_norm),
            Conv(n_filters_1, n_filters_2, kernel_size=3, stride=2, padding=1, batch_norm=batch_norm),
            Identity(),
            Conv(n_filters_2, n_filters_2, kernel_size=3, batch_norm=batch_norm),
            Conv(n_filters_2, n_filters_2, kernel_size=3, batch_norm=batch_norm),
            Conv(n_filters_2, n_filters_2, kernel_size=3, stride=2, padding=1, batch_norm=batch_norm),
            Identity(),
            Conv(n_filters_2, n_filters_2, kernel_size=3, batch_norm=batch_norm),
            Conv(n_filters_2, n_filters_2, kernel_size=1, batch_norm=batch_norm),    
            nn.AvgPool2d(8),
            Flatten(),
        )
        self.classifier = nn.Sequential(nn.Linear(n_filters_2, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def init_linear_like_repo(layer):
    if isinstance(layer, nn.Linear):
        nn.init.xavier_normal_(layer.weight)
        if layer.bias is not None:
            nn.init.zeros_(layer.bias)


shape_model = AllCNN().to(DEVICE)
shape_model.apply(init_linear_like_repo)
shape_model.eval()

with torch.no_grad():
    sample_images = torch.randn(2, 3, 32, 32, device=DEVICE)
    sample_logits = shape_model(sample_images)

print("Device:", DEVICE)
print("Model:", shape_model.__class__.__name__)
print("Sample input shape:", sample_images.shape)
print("Sample logits shape:", sample_logits.shape)
print("Total parameters:", sum(p.numel() for p in shape_model.parameters()))
assert sample_logits.shape == (2, NUM_CLASSES)

del shape_model, sample_images, sample_logits


In [ ]:

# ============================================================
# PHASE 4: TRAINING AND EVALUATION UTILITIES
# ============================================================
def sync_if_cuda():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total if total else float("nan")


def save_state_dict(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)


def load_state_dict_into(model, path):
    try:
        state = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    return model.to(DEVICE)


def train_from_scratch(train_loader, label, checkpoint_path, evaluation_loader):
    sync_if_cuda()
    start = time.perf_counter()
    model = AllCNN(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=ORIGINAL_LR, momentum=MOMENTUM)

    best_acc = 0.0
    for epoch in range(ORIGINAL_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            if MAX_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_BATCHES_PER_EPOCH:
                break
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * y.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)

        epoch_loss = running_loss / total if total else float("nan")
        epoch_acc = correct / total if total else float("nan")
        test_acc = accuracy(model, evaluation_loader)
        if test_acc >= best_acc:
            best_acc = test_acc
            save_state_dict(model, checkpoint_path)
        print(f"[{label}] Epoch {epoch + 1}/{ORIGINAL_EPOCHS} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f} | Test: {test_acc:.4f}")

    sync_if_cuda()
    elapsed = time.perf_counter() - start
    # As in the author code, return the final model while saving the best checkpoint.
    print(f"[{label}] Training time: {elapsed:.2f}s")
    print(f"[{label}] Best checkpoint saved: {checkpoint_path} | Test: {best_acc:.4f}")
    return model, elapsed


In [ ]:
def create_or_load_model(label, loader, checkpoint_path, evaluation_loader):
    checkpoint_path = Path(checkpoint_path)
    if REUSE_CHECKPOINTS and checkpoint_path.exists():
        model = AllCNN(num_classes=NUM_CLASSES).to(DEVICE)
        load_state_dict_into(model, checkpoint_path)
        print(f"[{label}] Loaded checkpoint: {checkpoint_path}")
        return model, None
    result = train_from_scratch(loader, label, checkpoint_path, evaluation_loader)
    if not isinstance(result, tuple) or len(result) != 2:
        raise ValueError(f"train_from_scratch must return (model, elapsed), got {type(result)} with {len(result) if isinstance(result, tuple) else 'non-tuple'} result.")
    return result


tag_settings = {
    "profile": RUN_PROFILE,
    "data_root": str(DATA_ROOT),
    "implementation_version": "author_alignment_v1",
    "evaluation_dataset": "official_cifar10_test",
    "checkpoint_selection": "best_test_accuracy",
    "extra_exp": EXTRA_EXP,
    "curvature_lambda": CURVATURE_LAMBDA,
    "reweight_bias": REWEIGHT_BIAS,
    "reweight_slope": REWEIGHT_SLOPE,
    "method": METHODS_TO_RUN,
    "forget_class": FORGET_CLASS,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "original_epochs": ORIGINAL_EPOCHS,
    "original_lr": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_lr": UNLEARN_LR,
    "momentum": MOMENTUM,
    "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "max_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
}

tag_hash = hashlib.sha256(
    json.dumps(tag_settings, sort_keys=True).encode("utf-8")
).hexdigest()[:8]

run_tag = f"{RUN_PROFILE}_forget{FORGET_CLASS}_{tag_hash}"

original_checkpoint = CHECKPOINT_ROOT / f"original_{run_tag}.pt"
retrain_checkpoint = CHECKPOINT_ROOT / f"retrain_{run_tag}.pt"

print("Run tag:", run_tag)
print("Original checkpoint:", original_checkpoint)
print("Retrain checkpoint:", retrain_checkpoint)

original_model, original_train_seconds = create_or_load_model(
    "original",
    train_all_loader,
    original_checkpoint,
    test_loader,
)



In [ ]:
retrain_model, retrain_train_seconds = create_or_load_model(
    "retrain",
    train_retain_loader,
    retrain_checkpoint,
    test_retain_loader,
)

print("Original model ready:", original_model is not None)
print("Retrain model ready:", retrain_model is not None)
print("Original training time:", original_train_seconds)
print("Retrain training time:", retrain_train_seconds)


In [ ]:

# ============================================================
# PHASE 5: FGSM ATTACK AND BOUNDARY SHRINK
# ============================================================
def fgsm_cifar(x_norm, y, model, bound=FGSM_BOUND, normalization_mode=ATTACK_NORMALIZATION):
    if normalization_mode == "matched":
        mean_values, std_values = CIFAR_MEAN, CIFAR_STD
    elif normalization_mode == "repo_exact":
        mean_values, std_values = REPO_ATTACK_MEAN, REPO_ATTACK_STD
    else:
        raise ValueError(f"Unsupported normalization mode: {normalization_mode}")
    mean = torch.tensor(mean_values, dtype=x_norm.dtype, device=x_norm.device).view(1, 3, 1, 1)
    std = torch.tensor(std_values, dtype=x_norm.dtype, device=x_norm.device).view(1, 3, 1, 1)
    x_natural_raw = x_norm.detach() * std + mean
    x_leaf = x_norm.detach().clone().requires_grad_(True)
    model.eval()
    logits = model(x_leaf)
    loss = F.cross_entropy(logits, y)
    grad = torch.autograd.grad(loss, x_leaf, only_inputs=True)[0]
    
    x_adv_raw = x_leaf.detach() * std + mean + bound * grad.sign()
    delta = (x_adv_raw - x_natural_raw).clamp(-bound, bound)
    x_adv_raw = (x_natural_raw + delta).clamp(0.0, 1.0)
    x_adv_raw = torch.round(x_adv_raw * 255.0) / 255.0
    x_adv = (x_adv_raw - mean) / std
    return x_adv.detach()


def train_limit_reached(batch_index):
    return MAX_BATCHES_PER_EPOCH is not None and batch_index >= MAX_BATCHES_PER_EPOCH


# Enhanced Boundary Shrink helpers ported from the author's expand_exp.py.
def _find_z(model, inputs, targets, h):
    inputs.requires_grad_()
    # Preserve the author's forward calls, including BatchNorm updates.
    outputs = model(inputs)
    loss_z = nn.CrossEntropyLoss()(model(inputs), targets)
    loss_z.backward()
    grad = inputs.grad.data + 0.0
    norm_grad = grad.norm().item()
    z = torch.sign(grad).detach() + 0.0
    z = h * (z + 1e-7) / (z.reshape(z.size(0), -1).norm(dim=1)[:, None, None, None] + 1e-7)
    inputs.grad.zero_()
    model.zero_grad()
    return z, norm_grad


def curvature(model, inputs, targets, h=3.0, lambda_=4):
    z, norm_grad = _find_z(model, inputs, targets, h)
    inputs.requires_grad_()
    outputs_pos = model(inputs + z)
    outputs_orig = model(inputs)
    loss_pos = nn.CrossEntropyLoss()(outputs_pos, targets)
    loss_orig = nn.CrossEntropyLoss()(outputs_orig, targets)
    grad_diff = torch.autograd.grad(loss_pos - loss_orig, inputs, create_graph=True)[0]
    reg = grad_diff.reshape(grad_diff.size(0), -1).norm(dim=1)
    model.zero_grad()
    return torch.sum(lambda_ * reg) / float(inputs.size(0)), reg


def PM(logit, target):
    # Allocate on the logits' device instead of hard-coding CUDA.
    eye = torch.eye(logit.shape[1], device=logit.device)
    probs_GT = (logit.softmax(1) * eye[target]).sum(1).detach()
    top2_probs = logit.softmax(1).topk(2, largest=True)
    GT_in_top2_ind = (top2_probs[1] == target.view(-1, 1)).float().sum(1) == 1
    probs_2nd = torch.where(GT_in_top2_ind, top2_probs[0].sum(1) - probs_GT, top2_probs[0][:, 0]).detach()
    return probs_2nd - probs_GT


def weight_assign(logit, target, bias, slope):
    pm = PM(logit, target)
    reweight = ((pm + bias) * slope).sigmoid().detach()
    return reweight * 3


def run_boundary_shrink(original, forget_loader, checkpoint_path):
    sync_if_cuda()
    start = time.perf_counter()
    teacher = copy.deepcopy(original).to(DEVICE)
    teacher.eval()

    unlearned = copy.deepcopy(original).to(DEVICE)
    
    optimizer = torch.optim.SGD(unlearned.parameters(), lr=UNLEARN_LR, momentum=MOMENTUM)
    criterion = nn.CrossEntropyLoss()
    crossed = 0
    seen = 0
    update_steps = 0

    for epoch in range(UNLEARN_EPOCHS):
        for batch_index, (x, y) in enumerate(forget_loader):
            if train_limit_reached(batch_index):
                break
            x = x.to(DEVICE, non_blocking=PIN_MEMORY)
            y = y.to(DEVICE, non_blocking=PIN_MEMORY)
            x_adv = fgsm_cifar(x, y, teacher)
            
            with torch.no_grad():
                adv_logits = teacher(x_adv)
                pseudo_label = adv_logits.argmax(dim=1)

            crossed += (pseudo_label != y).sum().item()
            seen += y.numel()

            unlearned.train()
            optimizer.zero_grad(set_to_none=True)
            logits = unlearned(x)
            ori_loss = criterion(logits, pseudo_label)
            if EXTRA_EXP == "curv":
                ori_curv = curvature(original, x, y, h=0.9)[1]
                cur_curv = curvature(unlearned, x, y, h=0.9)[1]
                loss = ori_loss + CURVATURE_LAMBDA * torch.norm(ori_curv - cur_curv, p=2)
            elif EXTRA_EXP == "weight_assign":
                weight = weight_assign(adv_logits, pseudo_label, bias=REWEIGHT_BIAS, slope=REWEIGHT_SLOPE)
                loss = (F.cross_entropy(logits, pseudo_label, reduction="none") * weight).mean()
            else:
                loss = ori_loss
            loss.backward()
            optimizer.step()
            update_steps += 1

        print(f"[boundary_shrink] Epoch {epoch + 1}/{UNLEARN_EPOCHS}")

    sync_if_cuda()
    elapsed = time.perf_counter() - start
    save_state_dict(unlearned, checkpoint_path)
    stats = {
        "seconds": elapsed,
        "cross_rate": crossed / seen if seen else float("nan"),
        "update_steps": update_steps,
        "sample_seen": seen,
    }
    print(f"[boundary_shrink] Training time: {elapsed:.2f}s")
    print(f"[boundary_shrink] Boundary crossing rate: {stats['cross_rate']:.4f}")
    print(f"[boundary_shrink] Update steps: {update_steps}")
    print(f"[boundary_shrink] Samples seen: {seen}")
    print(f"[boundary_shrink] Checkpoint saved: {checkpoint_path}")
    return unlearned, stats


In [ ]:

# ============================================================
# BOUNDARY EXPANDING
# ============================================================
def run_boundary_expanding(original, forget_loader, checkpoint_path):
    sync_if_cuda()
    start = time.perf_counter()

    # --------------------------------------------------------
    # 1. COPY FEATURE EXTRACTOR
    # --------------------------------------------------------
    features = copy.deepcopy(original.features).to(DEVICE)
    features.eval()

    original_classifier = original.classifier[0]
    feature_width = original_classifier.in_features

    # --------------------------------------------------------
    # 2. EXPAND CLASSIFIER: NUM_CLASSES -> NUM_CLASSES + 1
    # --------------------------------------------------------
    widened_classifier = nn.Linear(
        feature_width,
        NUM_CLASSES + 1
    ).to(DEVICE)

    init_linear_like_repo(widened_classifier)

    with torch.no_grad():
        widened_classifier.weight[:NUM_CLASSES].copy_(
            original_classifier.weight
        )
        widened_classifier.bias[:NUM_CLASSES].copy_(
            original_classifier.bias
        )

    widened_model = nn.Sequential(
        features,
        widened_classifier
    ).to(DEVICE)

    # --------------------------------------------------------
    # 3. TRAIN ON FORGET SET WITH SHADOW LABEL
    # --------------------------------------------------------
    optimizer = torch.optim.SGD(
        widened_model.parameters(),
        lr=UNLEARN_LR,
        momentum=MOMENTUM
    )
    criterion = nn.CrossEntropyLoss()
    update_steps = 0

    for epoch in range(UNLEARN_EPOCHS):

        for batch_index, (x, y) in enumerate(forget_loader):
            if train_limit_reached(batch_index):
                break

            x = x.to(DEVICE, non_blocking=PIN_MEMORY)

            shadow_label = torch.full(
                (x.shape[0],),
                NUM_CLASSES,
                dtype=torch.long,
                device=DEVICE
            )



            logits = widened_model(x)
            widened_model.train()
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(logits, shadow_label)

            loss.backward()
            optimizer.step()

            update_steps += 1



        print(f"[boundary_expanding] Epoch {epoch + 1}/{UNLEARN_EPOCHS} ")

    # --------------------------------------------------------
    # 4. PRUNE SHADOW CLASS
    # --------------------------------------------------------
    pruned_classifier = nn.Linear(
        feature_width,
        NUM_CLASSES
    ).to(DEVICE)

    with torch.no_grad():
        pruned_classifier.weight.copy_(
            widened_classifier.weight[:NUM_CLASSES]
        )
        pruned_classifier.bias.copy_(
            widened_classifier.bias[:NUM_CLASSES]
        )

    pruned_model = nn.Sequential(
        features,
        pruned_classifier
    ).to(DEVICE)

    # --------------------------------------------------------
    # 5. SAVE CHECKPOINTS
    # --------------------------------------------------------

    widened_checkpoint = checkpoint_path.with_name(
        checkpoint_path.stem + "_widened.pt"
    )

    save_state_dict(widened_model, widened_checkpoint)
    save_state_dict(pruned_model, checkpoint_path)

    sync_if_cuda()
    elapsed = time.perf_counter() - start

    stats = {
        "seconds": elapsed,
        "update_steps": update_steps,
    }

    print(f"[boundary_expanding] Training time: {elapsed:.2f}s")
    print(f"[boundary_expanding] Update steps: {update_steps}")
    print(f"[boundary_expanding] Widened checkpoint saved: {widened_checkpoint}")
    print(f"[boundary_expanding] Pruned checkpoint saved: {checkpoint_path}")

    return pruned_model, stats

In [ ]:
#PHASE 6: RUN UNLEARNING METHOD
unlearned_models = {}
unlearn_stats = {}

for method_name in METHODS_TO_RUN:
    method_checkpoint = CHECKPOINT_ROOT / f"{method_name}_{run_tag}.pt"
    method_forget_loader = make_loader(train_forget_set, shuffle = True)

    if method_name == "boundary_shrink":
        model, stats = run_boundary_shrink(original_model, method_forget_loader, method_checkpoint)
    elif method_name == "boundary_expanding":
        model, stats = run_boundary_expanding(original_model, method_forget_loader, method_checkpoint)
    else:
        raise ValueError(f"Unsupported unlearning method: {method_name}")

    unlearned_models[method_name] = model
    unlearn_stats[method_name] = stats

print("Finish methods:", list(unlearned_models))

In [ ]:
# ============================================================
# PHASE 7: OFFICIAL CIFAR-10 TEST EVALUATION
# ============================================================
def evaluate_model_splits(model, include_train=True):
    result = {
        "test_all_acc": accuracy(model, test_loader),
        "test_retain_acc": accuracy(model, test_retain_loader),
        "test_forget_acc": accuracy(model, test_forget_loader),
    }

    if include_train:
        result["train_retain_acc"] = accuracy(model, train_retain_eval_loader)
        result["train_forget_acc"] = accuracy(model, train_forget_eval_loader)
    else:
        result["train_retain_acc"] = None
        result["train_forget_acc"] = None

    return result


models_to_evaluate = {
    "original": original_model,
    "retrain": retrain_model,
    **unlearned_models,
}

rows = []
baseline_train_times = {"original": original_train_seconds, "retrain": retrain_train_seconds}

for name, model in models_to_evaluate.items():
    row = {"model": name}
    row.update(evaluate_model_splits(model, include_train=EVALUATE_TRAIN_SPLITS))

    if name in baseline_train_times:
        row["training_seconds"] = baseline_train_times[name]
    else:
        row["training_seconds"] = None

    row["unlearning_seconds"] = unlearn_stats[name]["seconds"] if name in unlearn_stats else None
    row["fgsm_cross_rate"] = unlearn_stats[name].get("cross_rate") if name in unlearn_stats else None
    row["unlearning_update_steps"] = unlearn_stats[name].get("update_steps") if name in unlearn_stats else None

    rows.append(row)

for row in rows:
    print("\n", row["model"])
    print("    train retain:", "n/a" if row["train_retain_acc"] is None else f"{row['train_retain_acc']:.2%}")
    print("    train forget:", "n/a" if row["train_forget_acc"] is None else f"{row['train_forget_acc']:.2%}")
    print("    test retain:", f"{row['test_retain_acc']:.2%}")
    print("    test forget:", f"{row['test_forget_acc']:.2%}")
    print("    test all:", f"{row['test_all_acc']:.2%}")
    if row["training_seconds"] is not None:
        print("    training seconds:", f"{row['training_seconds']:.1f}")
    if row["unlearning_seconds"] is not None:
        print("    unlearning seconds:", f"{row['unlearning_seconds']:.1f}")
    if row["fgsm_cross_rate"] is not None:
        print("    FGSM cross rate:", f"{row['fgsm_cross_rate']:.2%}")

metrics_path = OUTPUT_ROOT / f"{run_tag}_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

config = {
    "run_tag_settings": tag_settings,
    "run_profile": RUN_PROFILE,
    "data_root": str(DATA_ROOT),
    "methods": METHODS_TO_RUN,
    "forget_class": FORGET_CLASS,
    "forget_class_name": CIFAR_CLASSES[FORGET_CLASS],
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "evaluation_dataset": "official_cifar10_test",
    "train_sample_count": len(train_fit_set),
    "test_sample_count": len(evaluation_dataset),
    "extra_exp": EXTRA_EXP,
    "curvature_lambda": CURVATURE_LAMBDA,
    "reweight_bias": REWEIGHT_BIAS,
    "reweight_slope": REWEIGHT_SLOPE,
    "original_epochs": ORIGINAL_EPOCHS,
    "original_learning_rate": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_learning_rate": UNLEARN_LR,
    "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "train_npy_shape": list(train_images.shape),
    "test_npy_shape": list(test_images.shape),
    "sample_submission_rows": len(sample_rows),
    "device": str(DEVICE),
}

config_path = OUTPUT_ROOT / f"{run_tag}_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("\nMetrics saved to:", metrics_path)
print("Configuration saved to:", config_path)
print("Checkpoint folder:", CHECKPOINT_ROOT)